<a href="https://colab.research.google.com/github/sambslam/PitchXAI-Assignment/blob/main/Task1_Voice_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1 - Real-Time Voice Agent (Whisper + Ollama + Kokoro + Pipecat on RunPod)

PitchX assignment, Task 1

This notebook documents a voice conversation agent built on RunPod. The goal was a phone-style agent where someone speaks, the agent transcribes it, an LLM generates a reply, and the reply is spoken back, with the round trip under 800ms.

The pipeline was built with Pipecat using faster-whisper for speech-to-text, a local Ollama model as the brain, and Kokoro for speech. This maps directly to PitchX's core product, the AI calling agent.

How to read this notebook. The agent runs on the RunPod pod. Google Colab is a separate machine and can't reach the pod, so the cells here document what was run on the pod rather than running live in Colab. The same commands can be run on the provided RunPod access during the interview.


## 1. Architecture

A voice agent is a loop with four parts. Three of them do the work and one ties them together.

Speech to text, using faster-whisper. This is the ears. It takes the incoming audio and turns it into text. faster-whisper is an optimized build of OpenAI's Whisper that runs the same model faster on the GPU.

The model, using Ollama. This is the brain. It takes the transcribed text and generates a reply. This is the same local Llama 3.1 8B used in the other tasks.

Text to speech, using Kokoro. This is the mouth. It takes the model's reply and turns it back into spoken audio. Kokoro is a small, fast TTS model, which is what low latency needs.

Pipecat. This is the framework that connects the three pieces into a real-time pipeline and handles the streaming, timing, and audio plumbing. You define the pipeline as an ordered list of stages and Pipecat moves data through them. The reason for using it is that the streaming and overlap it provides is what makes hitting a low latency target possible at all.

The flow is: audio in, transcribe, model, speak, audio out, then wait for the next thing the person says and repeat.


## 2. Environment

Same pod as Tasks 2 and 3.

| Component | Value |
|---|---|
| Platform | RunPod GPU Pod (On-Demand) |
| GPU | NVIDIA RTX 6000 Ada Generation, 48 GB VRAM |
| Driver / CUDA | 550.127.05 / CUDA 12.4 |
| Python | 3.12.3 |
| STT | faster-whisper (base model) |
| LLM | Ollama, llama3.1:8b |
| TTS | Kokoro (af_heart voice) |
| Framework | Pipecat 1.3.0 |
| Live transport | SmallWebRTC (smallwebrtc) |

> Note on the model. The task didn't specify a model size for Task 1, so the same llama3.1:8b from the other tasks was reused. A larger model would improve reply quality and could be swapped in via the Ollama service config, since the 48GB card has room.


## 3. Setup

Ollama was already installed from the earlier tasks. The Task 1 additions were the three voice pieces and Pipecat with its extras.


In [ ]:
# --- pod terminal ---
# audio pieces
pip install faster-whisper kokoro soundfile requests
apt-get update && apt-get install -y ffmpeg espeak-ng

# pipecat core plus the extras actually needed
pip install pipecat-ai
pip install --ignore-installed cryptography
pip install "pipecat-ai[webrtc]" "pipecat-ai[silero]" "pipecat-ai[kokoro]" "pipecat-ai[runner]"


The audio test file was recorded on a phone as an m4a and converted to the format Whisper handles most cleanly:


In [ ]:
# --- pod terminal ---
ffmpeg -i input.m4a -ar 16000 -ac 1 input.wav

## 4. Each piece on its own

Before wiring anything together, each of the three pieces was tested alone so any problem could be isolated to one component.

faster-whisper transcribed the recorded question correctly:

```
Detected language: en
Hi, I saw your website and I'm interested in your service.
Can you tell me how it works and what it costs?
```

Kokoro generated speech from text and saved a playable wav file. The Ollama model was already confirmed working from Tasks 2 and 3. So all three pieces functioned individually.


## 5. The Pipecat pipeline (bot.py)

This is the full live agent. It defines the three services, seeds the model with a PitchX phone-agent system prompt, lays them out in a Pipecat pipeline, and serves the agent over a WebRTC transport so a browser can talk to it.

A few details worth knowing about getting this running on Pipecat 1.3.0, The context class is `LLMContext`, the aggregator is built with `LLMContextAggregatorPair`, and the WebRTC transport imports from `pipecat.transports.smallwebrtc.transport`.


In [ ]:
import os
from loguru import logger
from pipecat.audio.vad.silero import SileroVADAnalyzer
from pipecat.pipeline.pipeline import Pipeline
from pipecat.pipeline.runner import PipelineRunner
from pipecat.pipeline.task import PipelineParams, PipelineTask
from pipecat.processors.aggregators.llm_context import LLMContext
from pipecat.processors.aggregators.llm_response_universal import LLMContextAggregatorPair
from pipecat.runner.types import RunnerArguments, SmallWebRTCRunnerArguments
from pipecat.services.kokoro.tts import KokoroTTSService
from pipecat.services.ollama.llm import OLLamaLLMService
from pipecat.services.whisper.stt import WhisperSTTService
from pipecat.transports.base_transport import TransportParams


SYSTEM_PROMPT = (
    "You are a friendly phone agent for PitchX, a company that builds AI calling "
    "agents and automations for sales teams. You are speaking to someone who called "
    "in to learn about the product. Keep replies short and conversational, one or two "
    "sentences, since this is a voice call. Be warm, clear, and helpful. If you do not "
    "know a specific detail, say so simply rather than making it up."
)


async def run_bot(transport):
    stt = WhisperSTTService(model="base", device="cuda", compute_type="float16")
    llm = OLLamaLLMService(model="llama3.1:8b", base_url="http://localhost:11434/v1")
    tts = KokoroTTSService(voice_id="af_heart")

    context = LLMContext(messages=[{"role": "system", "content": SYSTEM_PROMPT}])
    context_aggregator = LLMContextAggregatorPair(context)

    pipeline = Pipeline([
        transport.input(),
        stt,
        context_aggregator.user(),
        llm,
        tts,
        transport.output(),
        context_aggregator.assistant(),
    ])

    task = PipelineTask(
        pipeline,
        params=PipelineParams(enable_metrics=True, enable_usage_metrics=True),
    )

    @transport.event_handler("on_client_connected")
    async def on_client_connected(transport, client):
        logger.info("Client connected, greeting them.")
        await task.queue_frames([context_aggregator.user().get_context_frame()])

    @transport.event_handler("on_client_disconnected")
    async def on_client_disconnected(transport, client):
        logger.info("Client disconnected.")
        await task.cancel()

    runner = PipelineRunner(handle_sigint=False)
    await runner.run(task)


async def bot(runner_args: RunnerArguments):
    transport = None
    if isinstance(runner_args, SmallWebRTCRunnerArguments):
        from pipecat.transports.smallwebrtc.transport import SmallWebRTCTransport
        transport = SmallWebRTCTransport(
            params=TransportParams(
                audio_in_enabled=True,
                audio_out_enabled=True,
                vad_analyzer=SileroVADAnalyzer(),
            ),
            webrtc_connection=runner_args.webrtc_connection,
        )
    else:
        logger.error(f"Unsupported runner arguments type: {type(runner_args)}")
        return
    await run_bot(transport)


if __name__ == "__main__":
    from pipecat.runner.run import main
    main()


Launched with the dev runner, binding to all interfaces so RunPod's proxy can reach it:

```bash
python3 bot.py --host 0.0.0.0 --port 7860 --proxy <pod-id>-7860.proxy.runpod.net
```

The server starts, serves a prebuilt web client, and the browser connects to it. The browser registers the microphone, the WebRTC signaling completes, and the audio tracks are received by the pod. Every part of this works.


## 6. What didn't work, and why

The live browser connection got most of the way and then stopped at one specific point. The page loads, the mic is picked up, the WebRTC signaling handshake completes, and the pod receives the audio tracks. But the connection never moves past "connecting" to "connected," and the logs fill with repeated `socket.send() raised exception` warnings.

The cause is NAT traversal. WebRTC needs a direct media path between the browser and the pod. The ICE candidates each side offered were private addresses (type host) and STUN-discovered addresses (type srflx). None of those were reachable across the two networks, so the media path could never form. There were no relay candidates, which are what a TURN server provides.

In plain terms: the browser and the pod are both behind NAT, RunPod's HTTP proxy isn't built to pass WebRTC media, and without a TURN server to relay the audio, the two ends can't reach each other. This is a known WebRTC behaviour, not a problem with the pipeline.

A TURN server was set up (free credentials from Metered.ca) and the attempt to inject it ran into a second wall: the Pipecat development runner creates the WebRTC connection internally, and the method to set ICE servers on that runner-created connection (`update_ice_servers`) doesn't exist on the connection object the runner hands the bot. So there was no clean way, within the dev runner, to attach the TURN server. Wiring TURN in properly would mean running a custom server instead of the dev runner, or using a transport like Daily that handles NAT traversal for you.

The decision was to stop chasing the live connection there and demonstrate the agent a different way, since the blocker is infrastructure plumbing rather than the agent itself.


## 7. The working demo (file-based)

To show the agent actually responds, the same three pieces were run in sequence on the recorded audio file, entirely on the pod, with no network involved. This sidesteps the WebRTC problem completely.

The flow is the same as the live pipeline, just driven by a file and run as plain sequential steps: audio file, transcribe with Whisper, send the text to Ollama, speak the reply with Kokoro, write the spoken reply to a wav file.


In [ ]:
import time
import requests
import soundfile as sf
from faster_whisper import WhisperModel
from kokoro import KPipeline

INPUT_AUDIO = "/workspace/input.wav"
OUTPUT_AUDIO = "/workspace/agent_reply.wav"

SYSTEM_PROMPT = (
    "You are a friendly phone agent for PitchX, a company that builds AI calling "
    "agents and automations for sales teams. Keep replies short and conversational, "
    "one or two sentences, since this is a voice call. Be warm and clear."
)


def transcribe(path):
    model = WhisperModel("base", device="cuda", compute_type="float16")
    segments, info = model.transcribe(path)
    return " ".join(seg.text for seg in segments).strip()


def get_reply(user_text):
    resp = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": "llama3.1:8b",
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_text},
            ],
            "stream": False,
        },
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["message"]["content"].strip()


def speak(text, path):
    pipeline = KPipeline(lang_code="a")
    for i, (gs, ps, audio) in enumerate(pipeline(text, voice="af_heart")):
        sf.write(path, audio, 24000)
    return path


def main():
    user_text = transcribe(INPUT_AUDIO)
    print("Heard:", user_text)
    reply = get_reply(user_text)
    print("Reply:", reply)
    speak(reply, OUTPUT_AUDIO)
    print("Saved spoken reply to", OUTPUT_AUDIO)


if __name__ == "__main__":
    main()


### Result

The agent took the spoken question and gave a correct, on-brand spoken reply.

```
Heard (STT, 1.37s):
    Hi, I saw your website and I'm interested in your service.
    Can you tell me how it works and what it costs?

Agent reply (LLM, 11.53s):
    Our AI calling agents can automate follow-ups, book demos, and even
    qualify leads for your sales team, all while providing a seamless
    customer experience. We'd be happy to schedule a demo to show you
    exactly how our system can work for you. Would you like to set that
    up for today or tomorrow?

Spoke reply (TTS, 6.99s) -> /workspace/agent_reply.wav
```

So the full loop works: speech in, an intelligent on-topic spoken response out.


## 8. On the 800ms target

The file demo measured about 20 seconds total, well above 800ms. There are two reasons for that, and both are expected.

A lot of the time is one-time model loading. The script loads each model fresh every run, including a download of a Kokoro component. With the models already loaded, the per-turn time drops a lot.

The bigger reason is that the demo runs the three stages one after another. It transcribes fully, then generates the full reply, then speaks it. I had to run it this way because the live streaming connection wasn't working, so the file version is sequential rather than overlapped. Streaming, where the model starts speaking before the reply is finished, is what brings the total down toward the target, and that depends on the live connection being up.

So the latency work is tied to fixing the connection. Once that's running, the usual levers apply: short replies, small fast models, and the GPU. The file demo shows the agent works; the streaming version is what makes it fast.


## 9. Where Task 1 stands

What works:
- All three components (Whisper, Ollama, Kokoro) work, individually and chained.
- A file-based demo runs the full loop and produces a correct spoken reply.
- The full Pipecat live pipeline is built and launches. It serves the web client, accepts a browser connection, registers the mic, completes signaling, and receives audio.

What's blocked:
- The live browser connection can't complete its media path on RunPod without a TURN server, and the dev runner doesn't expose a clean way to attach one. This is an infrastructure limitation, not a pipeline problem.

Next steps to make the live version work:
- Run the bot behind a transport that handles NAT traversal (such as Daily), or run a custom server that lets TURN credentials be attached to the connection, or deploy somewhere WebRTC media is handled for you.
- Once the live connection is up, measure and tune latency toward the 800ms target using the streaming overlap the pipeline already supports.

This notebook is documentation. Live execution is done on RunPod.
